# Segment trajectory plots

Notebook for plotting SubT-MRS (or any) trajectories against ground truth
**up to a time cutoff**, SE(3)-aligned, with per-method ATE in the legend.

Motivation: `low_light1` has camera pointing towards a person from ~735s onvwards that wrecks the full-sequence ATE for several front-ends (dynamic-object divergence). Trimming to the cutoff isolates the
pre-event accuracy. Configure `SEG_*` in the last cell.

In [ ]:
import os, copy, tempfile
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from evo.tools import file_interface
from evo.core import sync, metrics

REPO     = Path("/home/ubuntu/dl-vins-factory")
MAX_DIFF = 0.05      # GT<->est association window [s]
MIN_POSES = 5

GT_DIR = {
    "subt_mrs":       REPO / "dataset/6_SubT-MRS/groundtruth",
    "euroc":          REPO / "dataset/2_EuRoC/groundtruth",
    "ntu_viral":      REPO / "dataset/7_NTU-VIRAL/groundtruth",
    "botanic_garden": REPO / "dataset/8_Botanic-Garden/groundtruth",
}

COLORS = {
    "gftt_cpu": "#0062ff",
    "aliked_lk": "#ff9a60", "aliked_lightglue": "#ff4400",
    "superpoint_lk": "#7bff9a", "superpoint_lightglue": "#00ca1b",
    "raco_lk": "#ff7ad1", "raco_lightglue": "#ff00d4",
    "xfeat_lk": "#4ba9a4", "xfeat_lightglue": "#0082ae",
}
def color_for(m): return COLORS.get(m, "#705555")

def gt_path(ds, seq): return str(GT_DIR[ds] / f"{seq}_gt.tum")

def compute_ate(gt, est):
    """SE(3) Umeyama (no scale) translation ATE. -> (rmse, n, aligned_est, glen)."""
    ref = file_interface.read_tum_trajectory_file(gt)
    e   = file_interface.read_tum_trajectory_file(est)
    ref, e = sync.associate_trajectories(ref, e, max_diff=MAX_DIFF)
    if ref.num_poses < MIN_POSES:
        return np.nan, ref.num_poses, None, 0.0
    ea = copy.deepcopy(e); ea.align(ref, correct_scale=False)
    ape = metrics.APE(metrics.PoseRelation.translation_part); ape.process_data((ref, ea))
    return (ape.get_statistic(metrics.StatisticsType.rmse), ref.num_poses, ea,
            float(getattr(ref, "path_length", 0.0)))

def trim_tum(path, cut_abs):
    """Temp TUM with poses at timestamp <= cut_abs (time-sorted). cut_abs=None -> full."""
    # normalize each row to exactly 8 space-joined fields (drops headers / trailing
    # delimiters that evo's strict TUM reader rejects, e.g. some raw-VIO outputs)
    lines = []
    for l in Path(path).read_text().splitlines():
        p = l.split()
        if len(p) >= 8:
            try:
                float(p[0])
            except ValueError:
                continue
            lines.append(" ".join(p[:8]))
    lines.sort(key=lambda l: float(l.split()[0]))
    if cut_abs is not None:
        lines = [l for l in lines if float(l.split()[0]) <= cut_abs]
    tf = tempfile.NamedTemporaryFile("w", suffix=".tum", delete=False)
    tf.write("\n".join(lines) + "\n"); tf.close()
    return tf.name
print("setup ready")


In [ ]:
# ── Configure here ───────────────────────────────────────────────────────────
RUN_DIR         = REPO / "results/x86_64/subt_mrs/subt_mrs_eval_mono_loop"  # low_light1 lives here (kp256 was smoke-only)
SEG_DATASET     = "subt_mrs"
SEG_SEQ         = "low_light1"
SEG_CUT_ELAPSED = 740.0          # seconds from GT start; None = full sequence
LOOP_FILE       = "vio_trajectory_run01.tum"       # loop-corrected (pose graph)
NOLOOP_FILE     = "vio_trajectory_raw_run01.tum"   # raw VIO (no loop)
SAVE_PNG        = None           # e.g. RUN_DIR / "low_light1_to735s.png"; None = just show
# Extra non-run baselines to overlay: (label, tum_path, color, tag)
SEG_EXTRA = [
    ("LET-VINS",
     REPO / "results/x86_64/subt_mrs/letnet_baseline/low_light1/low_light1_loop.tum",
     "#00aa00", "loop"),
]
# ─────────────────────────────────────────────────────────────────────────────
gt  = gt_path(SEG_DATASET, SEG_SEQ)
g0  = np.loadtxt(gt)[0, 0]
cut_abs = (g0 + SEG_CUT_ELAPSED) if SEG_CUT_ELAPSED else None
cut_lbl = f"trimmed from {SEG_CUT_ELAPSED:.0f}s onwards" if SEG_CUT_ELAPSED else "full"

def _seg_best(mdir):
    """Best of loop/noloop by SE(3) ATE over the trimmed window. -> (ate, path, tag) or None."""
    cands = []
    for tag, fn in (("loop", LOOP_FILE), ("no-loop", NOLOOP_FILE)):
        p = mdir / SEG_SEQ / fn
        if p.exists():
            try:
                tt = trim_tum(str(p), cut_abs)
                a = compute_ate(gt, tt)[0]; os.unlink(tt)
            except Exception:
                continue          # malformed / empty trajectory (e.g. xfeat_lg) -> skip
            if np.isfinite(a):
                cands.append((a, str(p), tag))
    return min(cands) if cands else None   # min by ATE

# enumerate methods; per method pick the smaller-ATE variant (loop vs noloop)
series = []   # (label, path, color, tag)
for mdir in sorted(RUN_DIR.glob("dlvins_*_mono_loop")):
    method = mdir.name.replace("dlvins_", "").replace("_mono_loop", "")
    bv = _seg_best(mdir)
    if bv:
        a, p, tag = bv
        series.append((method, p, color_for(method), tag))
for lbl, p, c, tag in SEG_EXTRA:
    if Path(p).exists():
        series.append((lbl, str(p), c, tag))

fig, ax = plt.subplots(figsize=(9, 8))
gt_trim = trim_tum(gt, cut_abs)
refp = file_interface.read_tum_trajectory_file(gt_trim).positions_xyz
ax.plot(refp[:, 0], refp[:, 1], color="k", lw=2.4, label="GT", zorder=1)
ax.scatter(refp[0, 0], refp[0, 1], color="#000000", marker="o", s=80, zorder=6, label="GT Start")
ax.scatter(refp[-1, 0], refp[-1, 1], color="#555555", marker="X", s=80, zorder=6, label="GT End")
os.unlink(gt_trim)

for label, traj, c, tag in series:
    tt = trim_tum(traj, cut_abs)
    rmse, n, ea, glen = compute_ate(gt, tt); os.unlink(tt)
    if ea is None:
        print(f"  {label}: too few synced poses"); continue
    P = ea.positions_xyz
    ax.plot(P[:, 0], P[:, 1], color=c, lw=1.1, alpha=0.9, label=f"{label} [{tag}]  {rmse:.2f} m")

ax.set_xlim(-65, 5)
ax.set_ylim(-45, 15)

ax.set_aspect("equal", adjustable="datalim")
ax.set_xlabel("x [m]"); ax.set_ylabel("y [m]"); ax.grid(alpha=0.3)
ax.legend(fontsize=8, loc="best")
if SEG_DATASET == 'subt_mrs':
    dataset_name = 'SubT_MRS'
else:
    dataset_name = SEG_DATASET

dataset_name = dataset_name.replace('_', r'\_')
sequence_name = SEG_SEQ.replace('_', r'\_')
ax.set_title(f"$\\mathbf{{{dataset_name}}}$ - $\\mathbf{{{sequence_name}}}$\n" 
             "Trajectory (x-y, SE3-aligned to GT, best of loop/no-loop)\n"
             f"Pose {cut_lbl}")
fig.tight_layout()
if SAVE_PNG:
    fig.savefig(str(SAVE_PNG), dpi=140); print("saved", SAVE_PNG)
plt.show()


## Origin-aligned segment plot (rigid first-pose, no Umeyama)

Aligns each estimate to GT by the **first associated pose only** (origin + heading), with
**no SE(3) Umeyama and no scale** — so the clean pre-event segment overlaps GT exactly
and you can see *where* each method starts to diverge. An `X` marks the first pose whose
error exceeds `OA_DIV_THRESH` (divergence onset). Defaults to `smoke_handheld` raw VIO.


In [ ]:
# ── Origin-aligned segment plot (rigid first-pose align, NO Umeyama) ─────────────
# Aligns each estimate to GT by the FIRST associated pose only (origin + heading),
# no SE(3) Umeyama, no scale. Per method, plots whichever of loop/noloop has the
# SMALLER SE(3) ATE (tagged in the legend).
import copy
from evo.core import sync

OA_DATASET     = "subt_mrs"
OA_SEQ         = "smoke_handheld"
OA_RUN_DIR     = REPO / "results/x86_64/subt_mrs/subt_mrs_eval_mono_loop_kp256"
OA_LOOP_FILE   = "vio_trajectory_run01.tum"       # loop-corrected (pose graph)
OA_NOLOOP_FILE = "vio_trajectory_raw_run01.tum"   # raw VIO (no loop)
OA_DIV_THRESH  = 1.0     # m: error above this -> mark divergence onset
OA_CLIP_PAD    = 12.0    # m padding around GT extent (a blowup won't squash the view)
OA_EXTRA       = []      # optional [(label, tum_path, color)]

gt  = gt_path(OA_DATASET, OA_SEQ)
ref = file_interface.read_tum_trajectory_file(gt)
skip_method = ["aliked_lightglue", "raco_lightglue", "raco_lk", "xfeat_lightglue", "xfeat_lk", "superpoint_lightglue"]

def _best_variant(mdir):
    """Pick whichever of loop/no-loop has the smaller SE(3) ATE. -> (ate, path, tag) or None."""
    cands = []
    for tag, fn in (("loop", OA_LOOP_FILE), ("no-loop", OA_NOLOOP_FILE)):
        p = mdir / OA_SEQ / fn
        if p.exists():
            try:
                a = compute_ate(gt, str(p))[0]
            except Exception:
                continue          # malformed / empty trajectory -> skip
            if np.isfinite(a):
                cands.append((a, str(p), tag))
    # return min(cands) if cands else None   # min by ATE (first element)
    return cands[1] if cands else None

series = []   # (label, path, color, tag, ate)
for mdir in sorted(OA_RUN_DIR.glob("dlvins_*_mono_loop")):
    method = mdir.name.replace("dlvins_", "").replace("_mono_loop", "")
    if method in skip_method:
        continue
    bv = _best_variant(mdir)
    if bv:
        a, p, tag = bv
        series.append((method, p, color_for(method), tag, a))
for lbl, p, c in OA_EXTRA:
    if Path(p).exists():
        a = compute_ate(gt, str(p))[0]
        series.append((lbl, str(p), c, "extra", a))

fig, ax = plt.subplots(figsize=(10, 9))
ax.plot(ref.positions_xyz[:, 0], ref.positions_xyz[:, 1], "k-", lw=2.4, label="GT", zorder=1)
ax.scatter(ref.positions_xyz[0, 0], ref.positions_xyz[0, 1], c="k", s=70, zorder=10)

print(f"{OA_SEQ} - divergence onset (error > {OA_DIV_THRESH:.0f} m), origin-aligned; best of loop/noloop:")
for label, est, c, tag, a_se3 in series:
    try:
        e = file_interface.read_tum_trajectory_file(est)
        r, ea = sync.associate_trajectories(ref, e, max_diff=MAX_DIFF)
        ea = copy.deepcopy(ea); ea.align_origin(r)        # rigid first-pose align only
    except Exception as ex:
        print(f"  {label:22s} SKIP ({type(ex).__name__})"); continue
    P, G = ea.positions_xyz, r.positions_xyz
    err = np.linalg.norm(P - G, axis=1)
    lab = f"{label} [{tag}]  {a_se3:.2f} m" if np.isfinite(a_se3) else f"{label}  (nan)"
    ax.plot(P[:, 0], P[:, 1], "-", color=c, lw=1.0, alpha=0.9, label=lab)
    bad = np.where(err > OA_DIV_THRESH)[0]
    if len(bad):
        i = bad[0]; t_rel = r.timestamps[i] - r.timestamps[0]
        ax.scatter(P[i, 0], P[i, 1], color=c, s=60, marker="X", edgecolor="k", zorder=11)
        print(f"  {label:22s} [{tag:6s}] t={t_rel:6.0f}s  (pose {i})  final err {err[-1]:.1f} m")
    else:
        print(f"  {label:22s} [{tag:6s}] never exceeds {OA_DIV_THRESH:.0f} m  (max {err.max():.2f} m)")

gx, gy = ref.positions_xyz[:, 0], ref.positions_xyz[:, 1]
ax.set_xlim(gx.min() - OA_CLIP_PAD, gx.max() + OA_CLIP_PAD)
ax.set_ylim(gy.min() - OA_CLIP_PAD, gy.max() + OA_CLIP_PAD)
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("x [m]"); ax.set_ylabel("y [m]"); ax.grid(alpha=0.3)
ax.legend(fontsize=8, loc="best", ncol=2)
if OA_DATASET == 'subt_mrs':
    dataset_name = 'SubT_MRS'
else:
    dataset_name = OA_DATASET

dataset_name = dataset_name.replace('_', r'\_')
sequence_name = OA_SEQ.replace('_', r'\_')
ax.set_title(f"$\\mathbf{{{dataset_name}}}$ - $\\mathbf{{{sequence_name}}}$\n" 
             "Trajectory (x-y, origin-aligned to GT)\n"
             "X mark represent divergence onset")
fig.tight_layout(); plt.show()
